In [43]:
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama


# 1. Load the document
loader = TextLoader("company_info.txt", encoding="utf-8")
document = loader.load()


# 2. Chunking the document based on markdown format
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
chunks = splitter.split_text(document[0].page_content)


# 3. Loading an embedding model for Persian language
model_path = "./models/Tooka-SBERT-V2-Large"
embeddings = HuggingFaceEmbeddings(
    model_name="./models/Tooka-SBERT-V2-Large",
    model_kwargs={
        "device": "cpu",
        "local_files_only": True
    },
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32}
)


# 4. Build FAISS vector store
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("my_local_index")


# 5. Load local LLM via Ollama
llm = ChatOllama(model="DeepSeek-R1-Distill-Llama-8B-abliterated", temperature=0.0, num_gpu=-1, reasoning=True)


# 6. Define the prompt template
user_prompt = """
Answer the question based only on the following contexts:
If a part of the answer is not in the context, say: "متأسفانه اطلاعات کافی در اسناد موجود نیست, and respond the part that you know."
[context]: {context}
Question: {question}
[answer]:
"""
system_prompt = """You are a helpful Persian assistant acting as a Customer Service agent.
Answer questions in the user's language. Use formal emojis when appropriate.
CRITICAL RULES:
1. You MUST answer ONLY based on the provided context.
2. If a part of the answer is not in the context, say: "متأسفانه اطلاعات کافی در اسناد موجود نیست, and respond the part that you know."
3. DO NOT guess, infer, or use any outside knowledge.
4. Keep answers SHORT and precise.
"""
message = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", user_prompt),
])


# 7. Build the RAG chain using LCEL
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# 8. Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 9. create LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | message
    | llm
    | StrOutputParser()
)


# 10. Query
input_message = input("Enter your prompt: ")
response = rag_chain.invoke(input_message)
print(response)

Loading weights: 100%|██████████| 391/391 [00:01<00:00, 327.18it/s]


Enter your prompt:  introduce yourself.




سلام! من یک فارسیان است که به عنوان یک عامل خدمات مشتری عمل میکنم. میتوانم به زبان فارسی و زبانهای دیگر به شما کمک کنم. در حال حاضر در این نقش هستم و به شما در مورد هر موضوعی که نیاز دارید کمک میکنم. 😊
